# FIFA Men's World Cup 2026 Predictor
## Stage 2: Historical Feature Store Generation

**Author:** Kamweti Ryan Murunga  
**Objective:** Flatten the relational Fjelstul database tables into a unified team-level feature matrix. 

### Features to Engineered:
1. **Total Appearances:** Overall presence in historical tournaments.
2. **Attacking Strength:** Average goals scored per match across history.
3. **Defensive Vulnerability:** Average goals conceded per match across history.
4. **Winning DNA:** Historical tournament win ratio.

In [1]:
import pandas as pd
import numpy as np

# 1. Load the historical Fjelstul records
standings = pd.read_csv('data/group_standings.csv')

print(f"Loaded standings dataset with {len(standings)} historical group rows.")

# 2. Build the aggregation pipeline
feature_store = standings.groupby('team_name').agg(
    total_wc_appearances=('tournament_id', 'nunique'),
    total_games_played=('played', 'sum'),
    total_wins=('wins', 'sum'),
    total_goals_scored=('goals_for', 'sum'),
    total_goals_conceded=('goals_against', 'sum')
).reset_index()

# 3. Calculate core ratios safely to avoid division by zero
feature_store['hist_goals_per_game'] = feature_store['total_goals_scored'] / feature_store['total_games_played']
feature_store['hist_conceded_per_game'] = feature_store['total_goals_conceded'] / feature_store['total_games_played']
feature_store['hist_win_ratio'] = feature_store['total_wins'] / feature_store['total_games_played']

# 4. Drop the absolute sums to avoid scale bias, keeping only the normalized ratios
feature_store = feature_store[['team_name', 'total_wc_appearances', 'hist_goals_per_game', 'hist_conceded_per_game', 'hist_win_ratio']]

# 5. Save this as our standalone Feature Store file
feature_store.to_csv('data/engineered_team_features.csv', index=False)

print("\n[+] Feature Store successfully created and saved!")
display(feature_store.head(5))

Loaded standings dataset with 626 historical group rows.

[+] Feature Store successfully created and saved!


,team_name,total_wc_appearances,hist_goals_per_game,hist_conceded_per_game,hist_win_ratio
0,Algeria,4,1.000000,1.416667,0.250000
1,Angola,1,0.333333,0.666667,0.000000
2,Argentina,20,1.588235,1.441176,0.500000
3,Australia,13,1.256410,1.948718,0.256410
4,Austria,6,1.136364,1.272727,0.363636


In [1]:
import pandas as pd
import numpy as np

# 1. Load the standings table again
standings = pd.read_csv('data/group_standings.csv')

# 2. Extract Overall Baseline Stats (What we did before)
feature_store = standings.groupby('team_name').agg(
    total_wc_appearances=('tournament_id', 'nunique'),
    total_games_played=('played', 'sum'),
    total_wins=('wins', 'sum'),
    total_goals_scored=('goals_for', 'sum'),
    total_goals_conceded=('goals_against', 'sum')
).reset_index()

feature_store['hist_goals_per_game'] = feature_store['total_goals_scored'] / feature_store['total_games_played']
feature_store['hist_conceded_per_game'] = feature_store['total_goals_conceded'] / feature_store['total_games_played']
feature_store['hist_win_ratio'] = feature_store['total_wins'] / feature_store['total_games_played']

# 3. ADVANCED: Extract Modern Era Stats Only (Post-2010 World Cups)
# Modern tournaments: WC-2010, WC-2014, WC-2018, WC-2022
modern_ids = ['WC-2010', 'WC-2014', 'WC-2018', 'WC-2022']
modern_standings = standings[standings['tournament_id'].isin(modern_ids)]

modern_features = modern_standings.groupby('team_name').agg(
    modern_games=('played', 'sum'),
    modern_goals_scored=('goals_for', 'sum')
).reset_index()

modern_features['modern_goals_per_game'] = modern_features['modern_goals_scored'] / modern_features['modern_games']
modern_features = modern_features[['team_name', 'modern_goals_per_game']]

# 4. MERGE OVERALL AND MODERN FEATURES TOGETHER
advanced_feature_store = feature_store.merge(modern_features, on='team_name', how='left')

# If a team hasn't qualified since 2010, their modern goals will be NaN. Fill it with their overall historical average!
advanced_feature_store['modern_goals_per_game'] = advanced_feature_store['modern_goals_per_game'].fillna(advanced_feature_store['hist_goals_per_game'])

# 5. KEEP ONLY THE NORMALIZED RATIOS
final_features = advanced_feature_store[[
    'team_name', 'total_wc_appearances', 'hist_goals_per_game', 
    'hist_conceded_per_game', 'hist_win_ratio', 'modern_goals_per_game'
]]

# 6. OVERWRITE THE OLD FEATURE FILE WITH THE UPGRADED ONE
final_features.to_csv('data/engineered_team_features.csv', index=False)

print("[+] Upgraded Feature Store successfully created with Modern Form indexes!")
display(final_features.head(5))

[+] Upgraded Feature Store successfully created with Modern Form indexes!


,team_name,total_wc_appearances,hist_goals_per_game,hist_conceded_per_game,hist_win_ratio,modern_goals_per_game
0,Algeria,4,1.000000,1.416667,0.250000,1.000000
1,Angola,1,0.333333,0.666667,0.000000,0.333333
2,Argentina,20,1.588235,1.441176,0.500000,1.750000
3,Australia,13,1.256410,1.948718,0.256410,0.916667
4,Austria,6,1.136364,1.272727,0.363636,1.136364


In [3]:
import pandas as pd
import numpy as np

# 1. Load the original standings table
standings = pd.read_csv('data/group_standings.csv')

# 2. Base Aggregations (Including what we built before)
feature_store = standings.groupby('team_name').agg(
    total_wc_appearances=('tournament_id', 'nunique'),
    total_games_played=('played', 'sum'),
    total_wins=('wins', 'sum'),
    total_goals_scored=('goals_for', 'sum'),
    total_goals_conceded=('goals_against', 'sum')
).reset_index()

# Ratios
feature_store['hist_goals_per_game'] = feature_store['total_goals_scored'] / feature_store['total_games_played']
feature_store['hist_conceded_per_game'] = feature_store['total_goals_conceded'] / feature_store['total_games_played']
feature_store['hist_win_ratio'] = feature_store['total_wins'] / feature_store['total_games_played']

# 3. ADVANCED ERAS: Modern Recency Filter (Post-2010)
modern_ids = ['WC-2010', 'WC-2014', 'WC-2018', 'WC-2022']
modern_standings = standings[standings['tournament_id'].isin(modern_ids)]

modern_features = modern_standings.groupby('team_name').agg(
    modern_games=('played', 'sum'),
    modern_goals_scored=('goals_for', 'sum')
).reset_index()

modern_features['modern_goals_per_game'] = modern_features['modern_goals_scored'] / modern_features['modern_games']
modern_features = modern_features[['team_name', 'modern_goals_per_game']]

# 4. NEW DEFENSIVE METRICS: Clean Sheets (Assuming clean sheets data or proxy from group standings)
# We can approximate a defensive wall index by looking at low concession tracks
feature_store['defensive_wall_index'] = (feature_store['total_games_played'] - feature_store['hist_conceded_per_game']) / feature_store['total_games_played']

# Merge them all
advanced_store = feature_store.merge(modern_features, on='team_name', how='left')
advanced_store['modern_goals_per_game'] = advanced_store['modern_goals_per_game'].fillna(advanced_store['hist_goals_per_game'])

# 5. SELECT FINAL DIMENSIONS FOR EXPORT
final_features = advanced_store[[
    'team_name', 'total_wc_appearances', 'hist_goals_per_game', 
    'hist_conceded_per_game', 'hist_win_ratio', 'modern_goals_per_game', 'defensive_wall_index'
]]

# Save it out over the old one
final_features.to_csv('data/engineered_team_features.csv', index=False)
print("[+] Feature Store upgraded successfully with Defensive Stability metrics!")
display(final_features.head(3))

[+] Feature Store upgraded successfully with Defensive Stability metrics!


,team_name,total_wc_appearances,hist_goals_per_game,hist_conceded_per_game,hist_win_ratio,modern_goals_per_game,defensive_wall_index
0,Algeria,4,1.000000,1.416667,0.25,1.000000,0.881944
1,Angola,1,0.333333,0.666667,0.00,0.333333,0.777778
2,Argentina,20,1.588235,1.441176,0.50,1.750000,0.978806


In [4]:
import pandas as pd
import numpy as np

# 1. Load the core standings table
standings = pd.read_csv('data/group_standings.csv')

# 2. Re-build our strong baseline structural features
feature_store = standings.groupby('team_name').agg(
    total_wc_appearances=('tournament_id', 'nunique'),
    total_games_played=('played', 'sum'),
    total_wins=('wins', 'sum'),
    total_goals_scored=('goals_for', 'sum'),
    total_goals_conceded=('goals_against', 'sum'),
    total_points=('points', 'sum')
).reset_index()

# Ratios
feature_store['hist_goals_per_game'] = feature_store['total_goals_scored'] / feature_store['total_games_played']
feature_store['hist_conceded_per_game'] = feature_store['total_goals_conceded'] / feature_store['total_games_played']
feature_store['hist_win_ratio'] = feature_store['total_wins'] / feature_store['total_games_played']

# 3. NEW LEGAL FEATURE A: Tournament Points Intensity
# Captures how dominant a team is during group play over history
feature_store['points_per_game'] = feature_store['total_points'] / feature_store['total_games_played']

# 4. NEW LEGAL FEATURE B: Modern Recency Filter (2010 - 2022)
modern_ids = ['WC-2010', 'WC-2014', 'WC-2018', 'WC-2022']
modern_standings = standings[standings['tournament_id'].isin(modern_ids)]

modern_features = modern_standings.groupby('team_name').agg(
    modern_games=('played', 'sum'),
    modern_goals_scored=('goals_for', 'sum'),
    modern_wins=('wins', 'sum')
).reset_index()

modern_features['modern_goals_per_game'] = modern_features['modern_goals_scored'] / modern_features['modern_games']
modern_features['modern_win_ratio'] = modern_features['modern_wins'] / modern_features['modern_games']

# Keep only ratios for the merge
modern_features = modern_features[['team_name', 'modern_goals_per_game', 'modern_win_ratio']]

# 5. UIFIED MERGE & IMPUTATION FOR NEW WORLD CUP DEBUTANTS
final_features = feature_store.merge(modern_features, on='team_name', how='left')

# Impute historical metrics for debutants with 0, but handle form safely
final_features['modern_goals_per_game'] = final_features['modern_goals_per_game'].fillna(final_features['hist_goals_per_game'])
final_features['modern_win_ratio'] = final_features['modern_win_ratio'].fillna(final_features['hist_win_ratio'])

# Select features to expose to LightGBM
engineered_store = final_features[[
    'team_name', 'total_wc_appearances', 'hist_goals_per_game', 
    'hist_conceded_per_game', 'hist_win_ratio', 'points_per_game',
    'modern_goals_per_game', 'modern_win_ratio'
]]

# Save it over the disk file cleanly
engineered_store.to_csv('data/engineered_team_features.csv', index=False)
print("[+] Legal Feature Store successfully upgraded with Points Intensity & Recency Form arrays!")
display(engineered_store.head(3))

[+] Legal Feature Store successfully upgraded with Points Intensity & Recency Form arrays!


,team_name,total_wc_appearances,hist_goals_per_game,hist_conceded_per_game,hist_win_ratio,points_per_game,modern_goals_per_game,modern_win_ratio
0,Algeria,4,1.000000,1.416667,0.25,0.833333,1.000000,0.166667
1,Angola,1,0.333333,0.666667,0.00,0.666667,0.333333,0.000000
2,Argentina,20,1.588235,1.441176,0.50,1.426471,1.750000,0.750000
